# 📝 벡터 검색 과제 LV3 정답 — 사내 FAQ 시맨틱 검색봇 (강사용)

**모범답안 + 해설** 입니다. 경로는 `../../day15_RAG_벡터검색/data/` 입니다. 검색 함수는 하드웨어에 따라 미세하게 흔들릴 수 있어 **상위 1건의 분류(category)** 로 느슨하게 채점합니다. 메뉴 루프(문제 2)는 대화형이라 자가채점이 없습니다.

아래 셀을 먼저 실행해 라이브러리와 한국어 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 임베딩 모델을 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
(아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] FAQ 데이터를 살펴봅니다.
faqs = pd.read_csv('../../day15_RAG_벡터검색/data/faqs.csv')
print("행·열 크기:", faqs.shape)
print("\n[분류별 개수]"); print(faqs["category"].value_counts())
print("\n[앞 5행]"); display(faqs.head())

## 문제 1. FAQ 검색봇의 핵심 만들기 — 색인 + 검색 함수

검색봇의 심장은 두 부분입니다. **1) FAQ 질문들을 벡터로 색인**하고, **2) 사용자 질문과 가장 가까운 FAQ 1건을 찾는 함수**입니다. 아래 단계를 순서대로 완성하세요.

### 1단계 — 색인
- `faqs['question']` 전체를 임베딩해 변수 `faq_emb` 에 담으세요(`normalize_embeddings=True`).
- `chromadb.EphemeralClient()` 로 클라이언트를 열고, `get_or_create_collection('faq_bot', metadata={'hnsw:space': 'cosine'})` 로 컬렉션을 만들어 변수 `faq_bot` 에 담으세요.
- `faq_bot.add(...)` 로 20건을 넣으세요 — `ids=faqs['id'].tolist()`, `embeddings=faq_emb`, `documents=faqs['question'].tolist()`, `metadatas` 는 각 FAQ 의 `{'category':…, 'answer':…}` 리스트.

### 2단계 — 검색 함수
- `search_faq(question)` 함수를 정의하세요. 인자로 받은 질문 문자열을 임베딩해 `faq_bot.query(..., n_results=1)` 로 가장 가까운 FAQ 1건을 찾고, 그 **메타데이터 딕셔너리**(`res['metadatas'][0][0]`)를 돌려줍니다. 이 딕셔너리에는 `'category'` 와 `'answer'` 가 들어 있습니다.

**예시**
```
faq_bot.count()                              →  20
search_faq('택배가 어디쯤 왔는지 궁금해요')['category']  →  '배송'
search_faq('비밀번호를 까먹었어요')['category']         →  '회원'
```
<details><summary>힌트</summary>

```text
접근방법:
- 질문들을 임베딩해 컬렉션에 넣고, 사용자 질문을 임베딩해 가장 가까운 1건의 메타데이터를 돌려주는 함수를 만든다.

세부구현:
1. faqs 의 question 을 encode 로 faq_emb 에 담는다
2. EphemeralClient 로 클라이언트를 열고, 코사인 공간으로 컬렉션(faq_bot)을 만든다
3. add 에 ids·embeddings·documents·metadatas(category·answer) 를 넘긴다
4. search_faq 를 정의: 질문을 임베딩해 컬렉션에서 가장 가까운 1건을 query 로 찾고, 그 메타데이터 딕셔너리를 반환한다
```

</details>

In [ ]:
faq_emb = emb_model.encode(faqs['question'].tolist(), normalize_embeddings=True)
client = chromadb.EphemeralClient()
faq_bot = client.get_or_create_collection('faq_bot', metadata={'hnsw:space': 'cosine'})
faq_bot.add(
    ids=faqs['id'].tolist(),
    embeddings=faq_emb,
    documents=faqs['question'].tolist(),
    metadatas=[{'category': c, 'answer': a} for c, a in zip(faqs['category'], faqs['answer'])],
)

def search_faq(question):
    """사용자 질문과 가장 가까운 FAQ 1건의 메타데이터(category·answer)를 돌려줍니다."""
    qe = emb_model.encode([question], normalize_embeddings=True)
    res = faq_bot.query(query_embeddings=qe, n_results=1)
    return res['metadatas'][0][0]

print("색인된 FAQ 수:", faq_bot.count())
print("예시 검색:", search_faq("비밀번호를 까먹었어요"))

In [ ]:
# [자가채점]
assert faq_bot.count() == 20
assert search_faq('택배가 어디쯤 왔는지 궁금해요')['category'] == '배송'
assert search_faq('반품하면 환불금은 언제 돌려받나요')['category'] == '환불'
assert search_faq('비밀번호를 까먹었어요')['category'] == '회원'
print("✅ 문제1 통과!")

### 해설 — 문제 1
- **접근법**: 색인(인덱싱 타임)은 한 번만 해 두고, `search_faq` 는 질문이 올 때마다(쿼리 타임) 호출됩니다. `n_results=1` 이라 가장 가까운 한 건만 돌려줍니다.
- **흔한 실수**: `res['metadatas']` 는 **두 겹 리스트**입니다(질문 여러 개 × 결과 여러 개). 질문 하나·결과 하나이므로 `[0][0]` 으로 딕셔너리를 꺼냅니다.
- **대안**: `n_results` 를 늘려 상위 여러 건을 후보로 보여 준 뒤 사용자가 고르게 만들 수도 있습니다.

아래는 검색봇 메뉴에 쓸 **미리 준비된 질문 목록**입니다(제공 코드 — 실행만 하세요).

In [ ]:
# [제공 코드] 검색봇 메뉴에 보여 줄 예시 질문 6개
PRESET_QUESTIONS = [
    "택배가 어디쯤 왔는지 궁금해요",
    "반품하면 환불금은 언제 돌려받나요",
    "비밀번호를 까먹었어요",
    "다른 사이즈로 바꾸고 싶어요",
    "결제가 자꾸 실패해요",
    "쿠폰 두 개를 같이 쓸 수 있나요",
]
print("메뉴 질문 수:", len(PRESET_QUESTIONS))

## 문제 2. 대화형 메뉴 루프로 검색봇 완성하기

**배경**: 이제 문제 1 의 `search_faq` 와 위 `PRESET_QUESTIONS` 를 이용해, **번호를 고르면 답을 찾아 주는** 대화형 검색봇을 만듭니다.

### 1단계 — 선택 하나를 처리하는 함수 (자가채점 있음)
먼저 **번호 하나를 받아 결과 문자열을 돌려주는** 함수 `handle_choice(n)` 를 만드세요. 루프·입력과 떼어 놓으면 **눌러 보지 않고도 검사**할 수 있습니다(실무에서 대화형 프로그램을 테스트하는 방법이기도 합니다).

- `n` 이 `1`~`6` 이면 그 번호의 질문을 `search_faq` 로 찾아 `f"[{분류}] {답변}"` 형태의 **문자열을 돌려줍니다**(출력이 아니라 반환).
- `n` 이 그 밖의 값이면 문자열 `'없는 번호입니다.'` 를 돌려줍니다.

**예시**
```
handle_choice(1)   →  '[배송] …' 처럼 [분류] 로 시작하는 문자열
handle_choice(99)  →  '없는 번호입니다.'
```

### 2단계 — 메뉴 루프 (자가채점 없음 — 직접 눌러 보세요)
**요구사항** (아래 동작을 하는 코드를 답안 셀에 작성하세요):
- 검색 횟수를 셀 변수 `search_count` 를 `0` 으로 시작합니다(상태 관리).
- `while` 루프로 매번 **메뉴**를 출력합니다 — `PRESET_QUESTIONS` 를 `1`번부터 번호를 붙여 보여 주고, 마지막에 `0. 종료` 를 안내합니다.
- `int(input(...))` 로 번호를 받습니다.
  - `0` 이면 `f"검색봇을 종료합니다. (총 {search_count}번 검색)"` 를 출력하고 루프를 끝냅니다.
  - 그 밖의 번호면 **1단계의 `handle_choice(번호)` 결과를 출력**하고, 검색이 성공한 경우(즉 `없는 번호입니다.` 가 아닌 경우)에만 `search_count` 를 1 늘립니다.

> **종료 번호 0** 이 유일한 탈출구입니다(무한 루프가 되지 않게 반드시 0 분기에서 `break`).

> 이 문제는 자가채점이 없습니다. 직접 실행해 번호를 눌러 보고, `0` 으로 종료되는지 확인하세요.
<details><summary>힌트</summary>

```text
접근방법:
- 메뉴 출력 → 번호 입력 → 번호에 따라 분기(0=종료, 1~6=검색, 그 외=안내)를 while 로 반복한다.

세부구현:
1. search_count 를 0 으로 둔다
2. while 무한 반복 안에서:
   2-1. PRESET_QUESTIONS 를 enumerate(…, 1) 로 번호와 함께 출력하고 0. 종료 를 안내한다
   2-2. int(input(...)) 로 번호를 받는다
   2-3. 0 이면 종료 메시지를 출력하고 break
   2-4. 1~len 이면 그 질문을 search_faq 로 찾아 [분류] 답변 을 출력하고 search_count 를 1 늘린다
   2-5. 그 밖이면 없는 번호 안내
```

</details>

In [ ]:
def handle_choice(n):
    """번호 하나를 처리해 결과 문자열을 돌려준다(출력·입력과 분리)."""
    if 1 <= n <= len(PRESET_QUESTIONS):
        hit = search_faq(PRESET_QUESTIONS[n - 1])
        return f"[{hit['category']}] {hit['answer']}"
    return '없는 번호입니다.'

In [ ]:
# [자가채점]
_results = [handle_choice(i) for i in range(1, len(PRESET_QUESTIONS) + 1)]
# 번호마다 실제로 검색해 [분류] 와 답변 본문까지 돌려주는지
assert _results[0].startswith('[배송]'), \
    "1번(택배 위치)은 '[배송] ' 으로 시작하는 문자열이어야 합니다"
assert _results[2].startswith('[회원]'), \
    "3번(비밀번호)은 '[회원] ' 으로 시작해야 합니다"
assert _results[5].startswith('[쿠폰]'), \
    "6번(쿠폰 중복)은 '[쿠폰] ' 으로 시작해야 합니다"
assert all(len(r) > 20 for r in _results), \
    '분류만 돌려주지 말고 [분류] 뒤에 FAQ 답변 본문까지 이어 붙이세요'
assert len(set(_results)) == len(PRESET_QUESTIONS), \
    '번호마다 서로 다른 FAQ 답변이 나와야 합니다(고정 문자열을 돌려주면 안 됩니다)'
# 답변 본문이 실제 FAQ 데이터에서 온 것인지
assert all('] ' in r for r in _results), \
    "'[분류] 답변' 형태로 - 닫는 대괄호 뒤에 공백 하나를 두고 답변을 이어 붙이세요"
_answers = set(faqs['answer'])
assert all(r.split('] ', 1)[1] in _answers for r in _results), \
    'faqs 의 answer 를 그대로 붙여야 합니다'
assert handle_choice(99) == '없는 번호입니다.'
assert handle_choice(0) == '없는 번호입니다.'
print("✅ 문제2 1단계 통과!")

In [ ]:
search_count = 0
while True:
    print("\n[사내 FAQ 검색봇] 궁금한 항목의 번호를 고르세요.")
    for i, q in enumerate(PRESET_QUESTIONS, 1):
        print(f"  {i}. {q}")
    print("  0. 종료")
    choice = int(input("번호: "))
    if choice == 0:
        print(f"검색봇을 종료합니다. (총 {search_count}번 검색)")
        break
    result = handle_choice(choice)
    print("  ", result)
    if result != '없는 번호입니다.':
        search_count += 1

### 해설 — 문제 2
- **접근법**: `while True` 로 메뉴를 반복하되, `0` 을 받았을 때만 `break` 로 빠져나옵니다. `search_count` 처럼 루프 밖에서 만든 변수를 루프 안에서 갱신하는 것이 **상태 관리**입니다.
- **흔한 실수**: `break` 를 빠뜨리면 무한 루프가 됩니다. 또 `input()` 은 문자열을 주므로 `int(...)` 로 숫자로 바꿔야 번호 비교가 됩니다.
- **대안**: 잘못된 입력(숫자가 아닌 값)까지 막으려면 예외 처리를 쓸 수 있지만, 이 과정 범위에서는 숫자만 입력한다고 가정합니다.

---
## 문제 3. 분류를 좁혀 찾는 '고급 검색' 과 정확도 점검

**배경**: 지금 봇은 **전체 20건**에서 찾습니다. 그런데 사용자가 "배송 관련으로 물어볼 거예요"라고 미리 알려 주면, **그 분류 안에서만** 찾는 편이 더 정확합니다. 교안에서 배운 **메타데이터 필터(`where`)** 를 검색 함수에 붙이고, 정말 나아지는지 **숫자로 확인**합니다.

### 1단계 — 분류를 좁혀 찾는 함수
**요구사항**:
- 함수 `search_faq_in(question, category=None)` 을 만드세요.
- `category` 가 `None` 이면 지금처럼 **전체**에서 찾고, 값이 있으면 **그 분류 안에서만** 찾습니다(교안에서 배운 `where` 필터를 씁니다).
- 돌려주는 것은 문제 1 과 같은 **메타데이터 딕셔너리 1건**입니다.

**예시**
```
search_faq_in('언제 도착하나요')['category']            ->  '배송'
search_faq_in('언제 도착하나요', '환불')['category']     ->  '환불'  (환불 안에서만 찾으므로)
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 1 의 검색 함수에 조건 분기 하나를 더한다. 분류가 주어지면 컬렉션 조회에 필터를 얹는다.

세부구현:
1. 질문을 임베딩한다(문제 1 과 동일).
2. category 가 None 이면 필터 없이, 아니면 분류를 지정하는 필터를 넘겨 조회한다.
3. 두 경우 모두 메타데이터 1건을 돌려준다.
```

</details>

In [ ]:
def search_faq_in(question, category=None):
    """분류를 좁혀 FAQ 1건을 찾는다. category=None 이면 전체에서 찾는다."""
    qe = emb_model.encode([question], normalize_embeddings=True)
    if category is None:
        res = faq_bot.query(query_embeddings=qe, n_results=1)
    else:
        res = faq_bot.query(query_embeddings=qe, n_results=1,
                            where={'category': category})
    return res['metadatas'][0][0]

print('전체 검색:', search_faq_in('언제 도착하나요')['category'])
print('환불로 좁힘:', search_faq_in('언제 도착하나요', '환불')['category'])

In [ ]:
# [자가채점]
assert search_faq_in('언제 도착하나요')['category'] == '배송'
# 분류를 좁히면 반드시 그 분류 안에서만 나와야 한다
for cat in ['환불', '교환', '쿠폰']:
    assert search_faq_in('언제 도착하나요', cat)['category'] == cat, \
        f'{cat} 로 좁혔는데 다른 분류가 나왔습니다 - where 필터를 확인하세요'
print("✅ 3-1단계 통과!")

### 해설 — 문제 3 · 1단계
- **접근법**: 검색의 뼈대(임베딩 → 조회 → 메타데이터)는 그대로 두고 **필터만 조건부로** 얹습니다. `category=None` 을 기본값으로 두면 **기존 호출을 깨지 않고** 기능을 늘릴 수 있습니다.
- **흔한 실수**: 필터를 건 뒤에도 전체에서 찾을 거라 착각하는 것입니다. `where` 를 주면 **후보 자체가 그 분류로 줄어든** 뒤 그 안에서 가장 가까운 것을 고릅니다 — 그래서 배송 질문을 '환불'로 좁히면 환불 FAQ 중 가장 가까운 것이 나옵니다(엉뚱해 보여도 정상입니다).

### 2단계 — 좁히면 정말 나아질까 (정확도 점검)
필터가 **좋다는 말**로 끝내지 말고 숫자로 확인합니다. 아래 평가셋은 질문과 **정답 분류**의 짝입니다.

```python
EVAL = [('택배가 어디쯤 왔는지 궁금해요', '배송'),
        ('반품하면 환불금은 언제 들어오나요', '환불'),
        ('포인트는 어떻게 쓰나요', '회원'),
        ('사이즈가 안 맞아 다른 걸로 바꾸고 싶어요', '교환'),
        ('할인 코드는 어디에 넣나요', '쿠폰')]
```

**요구사항**:
- 위 `EVAL` 을 그대로 만드세요.
- **필터 없이** `search_faq_in(질문)` 으로 찾은 분류가 정답과 같은 개수를 `acc_plain` 에 담으세요(정수).
- **정답 분류로 좁혀** `search_faq_in(질문, 정답분류)` 로 찾았을 때 맞는 개수를 `acc_filtered` 에 담으세요(정수).
- 두 수를 나란히 출력하고, 어느 쪽이 높은지 한 줄로 적으세요.

> **생각해 볼 것**: 필터를 걸면 정확도가 100% 가 되는 게 당연해 보입니다. 그렇다면 실무에서 **항상 필터를 걸면 되는 것 아닐까요?** 아래 해설에서 함정을 확인하세요.
<details><summary>힌트</summary>

```text
접근방법:
- EVAL 을 한 번씩 돌며 두 가지 방식으로 각각 찾아 맞은 개수를 센다.

세부구현:
1. 두 카운터를 0 으로 시작한다.
2. for 로 (질문, 정답분류) 를 꺼내, 필터 없이 찾은 분류가 정답과 같으면 1 더한다.
3. 같은 질문을 정답분류로 좁혀 찾고, 같으면 다른 카운터에 1 더한다.
4. 두 값을 출력한다.
```

</details>

In [ ]:
EVAL = [('택배가 어디쯤 왔는지 궁금해요', '배송'),
        ('반품하면 환불금은 언제 들어오나요', '환불'),
        ('포인트는 어떻게 쓰나요', '회원'),
        ('사이즈가 안 맞아 다른 걸로 바꾸고 싶어요', '교환'),
        ('할인 코드는 어디에 넣나요', '쿠폰')]

acc_plain = 0
acc_filtered = 0
for q, gold in EVAL:
    if search_faq_in(q)['category'] == gold:
        acc_plain += 1
    if search_faq_in(q, gold)['category'] == gold:
        acc_filtered += 1

print(f'필터 없이   : {acc_plain}/{len(EVAL)}')
print(f'분류로 좁혀 : {acc_filtered}/{len(EVAL)}')
print('좁히면 후보가 줄어드니 당연히 높거나 같다 - 문제는 그 분류를 미리 알 수 있느냐다')

In [ ]:
# [자가채점]
assert len(EVAL) == 5
# 정수로 셌는지 (넘파이 정수로 누산해도 괜찮습니다)
assert int(acc_plain) == acc_plain and int(acc_filtered) == acc_filtered, \
    '맞은 개수는 정수여야 합니다'
assert acc_filtered == 5, '정답 분류로 좁히면 5개 모두 그 분류에서 나온다'
assert acc_plain == 4, \
    '필터 없이 찾으면 5개 중 4개가 맞습니다 - EVAL 을 그대로 만들고 실제로 세었는지 확인하세요'
assert acc_filtered >= acc_plain, '좁힌 쪽이 낮을 수는 없다'
print("✅ 3-2단계 통과! (필터 없이", acc_plain, "/ 좁혀서", acc_filtered, ")")

### 해설 — 문제 3 · 2단계 (함정 주의)
- **결과**: 필터 없이는 **4/5**('포인트는 어떻게 쓰나요' 가 '회원' 대신 '결제' 로 갔습니다), 정답 분류로 좁히면 **5/5** 입니다.
- **숫자는 이렇게 읽습니다**: `acc_filtered` 가 5/5 인 것은 검색이 똑똑해져서가 **아닙니다**. 정답 분류로 좁혔으니 **그 분류 밖의 답이 나올 수가 없어서** 5/5 입니다. 즉 이 실험은 **정답을 미리 알려 주고 맞혔는지 묻는** 셈입니다.
- **그래서 실무의 진짜 질문은** "필터를 걸까?" 가 아니라 "**분류를 어떻게 알아내지?**" 입니다. 사용자가 직접 고르게 하거나(드롭다운), 질문을 먼저 분류하는 단계를 두어야 합니다 — 그 분류가 틀리면 **정답이 후보에서 아예 빠져** 필터가 오히려 해가 됩니다.
- **흔한 실수**: 이런 평가를 만들어 놓고 "필터가 정확도를 올렸다"고 결론짓는 것입니다. **평가 설계가 결과를 정해 버리는** 대표적인 예이고, 다음 단원에서 배울 **검색 품질 지표**가 필요한 이유이기도 합니다.

---
## 문제 4. 검색봇에 답변 생성 붙이기 — 마지막 한 조각

> ⚠️ **이 문제는 `.env` 의 `OPENAI_API_KEY` 가 필요합니다.** 앞의 문제들과 달리 **실제 API 를 호출**하므로 **요금이 조금 듭니다**(정답 셀 1회 + 자가채점 1회 = 총 2회). 아래 제공 코드 셀을 먼저 실행하세요.

**배경**: 지금 검색봇은 FAQ 문서를 **그대로** 보여 줍니다. 딱딱하기도 하고, 고객이 물은 것과 초점이 어긋나 보일 때도 있습니다. 마지막으로 **찾은 FAQ 답변을 근거 삼아 고객에게 보낼 문장으로 다듬는** 단계를 붙여, 검색봇을 **RAG 봇**으로 완성합니다.

> 중요한 것은 **다듬되 지어내지 않는 것**입니다. 그래서 `system` 규칙으로 "주어진 FAQ 답변에 없는 내용은 절대 덧붙이지 마라" 를 못 박습니다.

In [ ]:
# [제공 코드] 답 생성에 쓸 OpenAI 클라이언트를 준비합니다 — 이 셀을 먼저 실행하세요.
# .env 파일에 OPENAI_API_KEY 를 넣어 두면 아래 한 줄이 그것을 읽어 연결합니다.
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('.env')                # 같은 폴더의 .env
load_dotenv('../../day15_RAG_벡터검색/.env')             # (정답 폴더처럼 한 단계 안에서 열었을 때)

# 키가 없으면 OpenAI() 를 만드는 순간 알아보기 어려운 에러가 납니다.
# 그래서 먼저 확인하고, 없으면 무엇을 해야 하는지 알려 줍니다.
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError(
        '이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n'
        '  1) 일차 폴더에서  cp .env.example .env\n'
        '  2) .env 를 열어 OPENAI_API_KEY 에 본인 키를 채우기\n'
        '  3) 커널을 다시 시작한 뒤 이 셀부터 실행'
    )

# max_retries : 분당 한도에 걸려 거절당하면(429) 잠시 뒤 자동으로 다시 시도할 횟수
client = OpenAI(max_retries=8)     # OPENAI_API_KEY 를 환경변수에서 자동으로 찾아 쓴다
print('연결 준비 완료 — 키 확인됨')

**요구사항**:
- 함수 `answer_with_llm(question)` 을 만드세요.
- 문제 1 의 `search_faq(question)` 로 **가장 가까운 FAQ 1건**을 찾습니다.
- 그 FAQ 의 `answer` 를 근거로, `client.chat.completions.create(model='gpt-4o-mini', messages=..., temperature=0, max_tokens=200)` 를 호출해 **고객에게 보낼 친절한 한두 문장**을 받아 **문자열로 돌려줍니다**(출력이 아니라 반환).
- `messages` 는 **`system`** 과 **`user`** 두 개로 만드세요.
  - `system`: 고객센터 상담원 역할 + "**주어진 FAQ 답변에 없는 내용은 절대 덧붙이지 마라**" + "기간·장소·절차 같은 **핵심 낱말은 원문 그대로** 남겨라".
  - `user`: 찾은 FAQ 답변과 고객 질문을 함께 넣습니다.

**예시**
```
answer_with_llm('주문한 상품은 언제쯤 도착하나요')
  →  '결제가 완료되면 영업일 기준 2~3일 안에 받아 보실 수 있어요. …' (매번 조금씩 달라집니다)
```
<details><summary>힌트</summary>

```text
접근방법:
- 검색은 이미 만든 함수를 쓰고, 그 결과의 답변을 프롬프트에 넣어 모델에게 다듬게 한 뒤 받은 문자열을 반환한다.

세부구현:
1. search_faq 로 질문에 가장 가까운 FAQ 메타데이터를 얻는다
2. 그 메타의 answer 를 꺼낸다
3. system 규칙 한 개와, FAQ 답변·고객 질문을 담은 user 메시지 한 개로 messages 를 만든다
4. chat.completions.create 를 호출하고 choices[0].message.content 를 반환한다
```

</details>

In [ ]:
def answer_with_llm(question):
    """검색한 FAQ 답변을 근거로, 고객에게 보낼 한두 문장을 만들어 돌려준다."""
    hit = search_faq(question)
    messages = [
        {'role': 'system',
         'content': '너는 고객센터 상담원이다. 아래 FAQ 답변에 있는 내용만으로 고객에게 보낼 '
                    '친절한 한두 문장을 쓴다. FAQ 답변에 없는 내용은 절대 덧붙이지 않고, '
                    '기간·장소·절차 같은 핵심 낱말은 원문 그대로 남긴다.'},
        {'role': 'user',
         'content': f"[FAQ 답변] {hit['answer']}\n\n[고객 질문] {question}"},
    ]
    return client.chat.completions.create(
        model='gpt-4o-mini', messages=messages, temperature=0, max_tokens=200
    ).choices[0].message.content

print(answer_with_llm('주문한 상품은 언제쯤 도착하나요'))

In [ ]:
# [자가채점]  LLM 답변은 매번 조금씩 달라지므로 '정확히 이 문장' 으로는 채점할 수 없습니다.
#            그래서 1) 근거로 쓴 FAQ 의 분류가 맞는지 2) 문자열이 실제로 돌아왔는지
#            3) 원본 FAQ 답변의 핵심 낱말이 살아 있는지(지어내거나 뭉개지 않았는지) 를 봅니다.
_q = '주문한 상품은 언제쯤 도착하나요'
assert search_faq(_q)['category'] == '배송', \
    '이 질문은 배송 FAQ 를 근거로 삼아야 합니다 - 문제 1 의 search_faq 를 확인하세요'
_reply = answer_with_llm(_q)          # 실제 API 호출 1회
assert isinstance(_reply, str) and len(_reply.strip()) >= 15, \
    'answer_with_llm 은 비어 있지 않은 문자열을 반환해야 합니다(print 가 아니라 return)'
_anchors = ['영업일', '2~3', '도서 산간']
assert any(_a in _reply for _a in _anchors), \
    f'원본 FAQ 답변의 핵심 낱말({_anchors}) 이 하나도 남지 않았습니다 - system 규칙에 '\
    '"핵심 낱말은 원문 그대로" 를 넣었는지 확인하고 다시 실행해 보세요'
print("✅ 문제4 통과!")
print('[생성된 답변]', _reply)

### 해설 — 문제 4
- **접근법**: 검색봇(찾기)에 **생성**을 한 겹 얹으면 RAG 봇이 됩니다. 구조는 늘 같습니다 — **찾은 문서를 프롬프트에 넣고, `system` 으로 '근거 밖으로 나가지 마라' 는 규칙을 준다.** `search_faq` 를 그대로 재사용했다는 점도 눈여겨보세요. 앞에서 만든 조각이 그대로 부품이 됩니다.
- **왜 정확 일치로 채점하지 않나**: LLM 출력은 같은 입력이라도 **매번 표현이 달라집니다** (`temperature=0` 이어도 완전히 같다고 보장되지 않습니다). 그래서 문장을 통째로 비교하지 않고 **성질**을 검사했습니다 — 1) 근거로 쓴 FAQ 의 분류가 '배송' 인가(검색이 제대로 됐는가) 2) 반환값이 비어 있지 않은 문자열인가(호출이 성공했고 `return` 했는가) 3) 원본 답변의 핵심 낱말('영업일'·'2~3'·'도서 산간') 중 하나라도 남아 있는가(내용을 지어내거나 뭉개지 않았는가). 3번은 여러 낱말 중 **하나만 있어도 통과**시켜, 표현이 흔들려도 억울하게 떨어지지 않게 했습니다.
- **흔한 실수**: `print` 만 하고 `return` 을 빠뜨리는 것입니다. 그러면 반환값이 `None` 이라 2번에서 걸립니다. 또 `system` 규칙을 빼면 모델이 '보통 2~3일 걸립니다' 처럼 **FAQ 에 없는 일반 상식**을 섞기 시작합니다 — 고객센터 봇에서는 그게 사고로 이어집니다.
- **대안**: `search_faq` 대신 `n_results` 를 3 으로 늘려 여러 FAQ 를 근거로 주면 애매한 질문에 더 잘 답합니다. 반대로 **가장 가까운 FAQ 도 충분히 가깝지 않으면**(거리가 크면) 생성으로 넘기지 않고 "상담원 연결" 로 빠지는 안전장치를 두는 것이 실무 패턴입니다.